<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/notebooks/07_evaluacion_ood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Evaluación fuera de distribución (OOD)

Este notebook evalúa el modelo mBERT entrenado en inglés (notebook 05)
contra los dos conjuntos de evaluación en español nativo: SpaPhish y
SpearPhishMX. Ninguno de los dos se usó en ninguna etapa de
entrenamiento ni validación — esta es la primera vez que el modelo los
ve, lo que permite medir su capacidad real de generalización
cross-lingual (entrenado en inglés, evaluado en español).

Se evalúan por separado, no combinados, para distinguir el desempeño
en phishing genérico (SpaPhish) frente a spear-phishing dirigido
(SpearPhishMX).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers datasets scikit-learn -q

## 1. Carga del modelo mBERT entrenado (inglés)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Dispositivo:', device)

RUTA_MODELO_MBERT = '/content/drive/MyDrive/modelo_mbert_es'  # AJUSTAR a la ruta real donde guardaste el modelo en inglés

tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO_MBERT)
modelo_mbert = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO_MBERT)
modelo_mbert = modelo_mbert.to(device)
modelo_mbert.eval()

print('Modelo mBERT cargado correctamente.')

## 2. Carga de los conjuntos de evaluación OOD

In [ ]:
import pandas as pd

RUTA_SPAPHISH = '/content/drive/MyDrive/spaphish_limpio.csv'
RUTA_SPEARPHISHMX = '/content/drive/MyDrive/spearphishmx_limpio.csv'

df_spaphish = pd.read_csv(RUTA_SPAPHISH)
df_spearphishmx = pd.read_csv(RUTA_SPEARPHISHMX)

print('SpaPhish:', df_spaphish.shape)
print(df_spaphish['label'].value_counts())
print()
print('SpearPhishMX:', df_spearphishmx.shape)
print(df_spearphishmx['label'].value_counts())

In [ ]:
df_spaphish = df_spaphish.drop(columns=['n_saltos'])
df_spearphishmx = df_spearphishmx.drop(columns=['n_saltos'])

print('Columnas SpaPhish:', df_spaphish.columns.tolist())
print('Columnas SpearPhishMX:', df_spearphishmx.columns.tolist())

## 3. Función de evaluación OOD

Se define una función reutilizable que tokeniza un conjunto con el
tokenizer del modelo, genera predicciones, y calcula las métricas
estándar (accuracy, precision, recall, F1) — solo inferencia, sin
entrenar nada.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

In [ ]:

from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np


def evaluar_ood_con_progreso(modelo, tokenizer, df, nombre_conjunto, batch_size=16):
    textos = df['text'].astype(str).tolist()
    etiquetas_reales = df['label'].tolist()

    predicciones = []
    modelo.eval()
    n_lotes = (len(textos) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in tqdm(range(0, len(textos), batch_size), total=n_lotes, desc=f"Evaluando {nombre_conjunto}"):
            lote = textos[i:i+batch_size]
            inputs = tokenizer(lote, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
            salidas = modelo(**inputs)
            preds_lote = torch.argmax(salidas.logits, dim=-1).cpu().numpy()
            predicciones.extend(preds_lote)

    accuracy = accuracy_score(etiquetas_reales, predicciones)
    precision, recall, f1, _ = precision_recall_fscore_support(
        etiquetas_reales, predicciones, average='binary'
    )

    print(f'--- Resultados OOD sobre {nombre_conjunto} ---')
    print(f'Accuracy : {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall   : {recall:.4f}')
    print(f'F1       : {f1:.4f}')
    print()

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

## 4. Evaluación sobre SpaPhish (phishing genérico en español)

In [ ]:
metricas_spaphish = evaluar_ood_con_progreso(modelo_mbert, tokenizer, df_spaphish, 'SpaPhish')

## 5. Evaluación sobre SpearPhishMX (spear-phishing dirigido en español)

In [ ]:
metricas_spearphishmx = evaluar_ood_con_progreso(modelo_mbert, tokenizer, df_spearphishmx, 'SpearPhishMX')

In [ ]:
import numpy as np

textos_muestra = df_spaphish['text'].astype(str).tolist()[:20]
inputs = tokenizer(textos_muestra, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
with torch.no_grad():
    salidas = modelo_mbert(**inputs)
preds = torch.argmax(salidas.logits, dim=-1).cpu().numpy()
print('Predicciones en 20 muestras de SpaPhish:', preds)
print('Logits crudos (primeras 3 filas):', salidas.logits[:3])

In [ ]:
# Comparamos directamente los pesos del modelo cargado contra
# lo que esperaríamos de un modelo SIN entrenar (recién inicializado)
for nombre, parametro in modelo_mbert.named_parameters():
    if 'classifier' in nombre:
        print(nombre, parametro.shape)
        print(parametro.flatten()[:5])
        print()

In [ ]:
import os
ruta = '/content/drive/MyDrive/modelo_mbert_en'
for archivo in os.listdir(ruta):
    ruta_completa = os.path.join(ruta, archivo)
    tamano_mb = os.path.getsize(ruta_completa) / (1024*1024)
    print(f'{archivo}: {tamano_mb:.1f} MB')